# 05 Evaluation Pipeline Starter

Project: **AI Support Operations SLA Breach and Priority Triage**

## Objective
Build a reusable pipeline, perform cross-validation, tune hyperparameters and save the final model artifact.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_auc_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

path = Path("../01_Data/processed data/model_ready_support_sla_sample.csv")
df = pd.read_csv(path, low_memory=False)

target = "sla_breached"
X = df.drop(columns=[target, "resolution_hours"], errors="ignore")
y = df[target]

numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string", "category", "bool"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


## Evaluation Pipeline Setup

The modelling dataset was reloaded to establish a reproducible evaluation workflow. Predictor variables and the target variable were separated, and the same train-test split strategy used during model development was applied to ensure consistency across evaluation stages.

Numerical and categorical features were identified automatically to support preprocessing within machine learning pipelines. This approach ensures that model validation, hyperparameter tuning, and future deployment steps use identical feature definitions and preprocessing logic.

The objective of this notebook is to evaluate model robustness, assess generalisation performance, and prepare a production-ready prediction pipeline.


## Build final pipeline

In [2]:
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ]
)

pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=-1))
])


## Evaluation Model Pipeline

A complete machine learning pipeline was constructed to ensure that preprocessing and model training occur within a single reproducible workflow. Numerical features were imputed and standardised, while categorical features were imputed and one-hot encoded.

A Random Forest classifier was selected for the evaluation pipeline due to its robustness, ability to model non-linear relationships, and support for feature importance analysis. The model was configured with balanced class weighting to account for the moderate class imbalance present within the SLA breach target.

Using a unified pipeline ensures that all preprocessing transformations are applied consistently during training, validation, and future deployment, reducing the risk of data leakage and improving reproducibility.


## Cross-validation

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["precision", "recall", "f1", "roc_auc"]
cv_results = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

summary = {metric: cv_results[f"test_{metric}"].mean() for metric in scoring}
summary.update({metric + "_std": cv_results[f"test_{metric}"].std() for metric in scoring})
summary


{'precision': 0.8924368424755528,
 'recall': 0.903504711182023,
 'f1': 0.8978650094907689,
 'roc_auc': 0.9585344068473827,
 'precision_std': 0.006996426396287763,
 'recall_std': 0.014921844605266504,
 'f1_std': 0.008335833458396134,
 'roc_auc_std': 0.004121960051509835}

## Cross-Validation Results

A five-fold stratified cross-validation procedure was performed to evaluate model stability and generalisation performance. Stratification ensured that the SLA breach class distribution remained consistent across all validation folds.

The Random Forest classifier achieved strong and consistent performance across all evaluation metrics. Average precision was 89.2%, recall was 90.4%, and the resulting F1-Score was 89.8%. The model also achieved an ROC-AUC score of 95.9%, demonstrating excellent ability to distinguish between breached and non-breached tickets.

The standard deviations across folds were very small, indicating that model performance remained stable regardless of the specific training and validation split. This suggests that the model generalises well and is unlikely to be overly sensitive to sampling variation.

These results provide confidence that the model can maintain strong predictive performance when applied to unseen support tickets in a production environment.


## Hyperparameter tuning

In [5]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 8, None],
    "model__min_samples_leaf": [1, 3, 5]
}

search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print("Best CV F1:", search.best_score_)


Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best params: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
Best CV F1: 0.8994059833300738


## Hyperparameter Optimisation

A Grid Search procedure was conducted to identify an improved set of Random Forest hyperparameters. Three key parameters were evaluated: the number of trees (`n_estimators`), maximum tree depth (`max_depth`), and minimum leaf size (`min_samples_leaf`).

The search evaluated 18 parameter combinations using five-fold stratified cross-validation, resulting in a total of 90 model fits. Model selection was based on the F1-Score because the business objective requires balancing the detection of SLA breaches with the avoidance of excessive false alarms.

The best-performing configuration used 200 trees, unrestricted tree depth, and a minimum leaf size of one observation. This configuration achieved a cross-validated F1-Score of 0.899.

Only a small improvement was observed compared with the baseline Random Forest model, suggesting that the original configuration was already performing near its optimal level. This indicates that predictive performance is driven primarily by feature quality and dataset characteristics rather than extensive hyperparameter tuning.


## Final evaluation and threshold discussion

In [8]:
final_model = search.best_estimator_
proba = final_model.predict_proba(X_test)[:, 1]
default_preds = (proba >= 0.50).astype(int)

print("Default threshold 0.50")
print(classification_report(y_test, default_preds, zero_division=0))
print(confusion_matrix(y_test, default_preds))
print("ROC-AUC:", roc_auc_score(y_test, proba))

# Compare thresholds for business trade-off
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
rows = []
for t in thresholds:
    preds = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    rows.append({"threshold": t, "TP": tp, "FP": fp, "FN": fn, "TN": tn, "F1": f1_score(y_test, preds)})
threshold_df = pd.DataFrame(rows)
display(threshold_df)


Default threshold 0.50
              precision    recall  f1-score   support

           0       0.86      0.82      0.84      1174
           1       0.89      0.91      0.90      1826

    accuracy                           0.88      3000
   macro avg       0.87      0.86      0.87      3000
weighted avg       0.87      0.88      0.87      3000

[[ 958  216]
 [ 159 1667]]
ROC-AUC: 0.9571038529213649


,threshold,TP,FP,FN,TN,F1
0,0.3,1819,481,7,693,0.881726
1,0.4,1770,361,56,813,0.894617
2,0.5,1667,216,159,958,0.898895
3,0.6,1518,89,308,1085,0.884358
4,0.7,1397,37,429,1137,0.857055


## Threshold Analysis and Business Trade-Offs

The final Random Forest model was evaluated across multiple probability thresholds to understand the trade-off between identifying SLA breaches and generating false alarms.

At lower thresholds, the model identifies a larger proportion of potential SLA breaches, resulting in higher recall but also producing more false positives. Conversely, higher thresholds reduce false positives but increase the number of missed SLA breaches.

The default threshold of 0.50 produced the highest F1-Score (0.899), indicating the strongest balance between precision and recall. At this threshold, the model correctly identified 1,667 SLA breaches while missing only 159 breach events.

From a business perspective, threshold selection depends on operational priorities. If the objective is to minimise missed SLA breaches, a lower threshold may be appropriate. If the objective is to reduce unnecessary escalations and false alarms, a higher threshold may be preferred.

For this project, a threshold of 0.50 is recommended because it provides the best overall balance between detection performance and operational efficiency.


## Save final pipeline

In [9]:
# =========================
# Save Final Trained Model
# =========================

from pathlib import Path
import joblib

# Use the best model from GridSearchCV
final_model = search.best_estimator_

# Create models directory if it does not exist
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

# Save model
model_path = MODEL_DIR / "final_sla_breach_pipeline.joblib"
joblib.dump(final_model, model_path)

print("Model saved successfully!")
print("Location:", model_path)

# Verify file exists
print("File exists:", model_path.exists())

Model saved successfully!
Location: ..\models\final_sla_breach_pipeline.joblib
File exists: True


## Model Persistence

The final optimised machine learning pipeline was saved using Joblib to support future deployment and reuse. Saving the complete pipeline ensures that all preprocessing steps, feature transformations, and model parameters remain consistent between training and production environments.

The saved pipeline includes data imputation, feature scaling, categorical encoding, and the trained Random Forest classifier. This approach eliminates the risk of preprocessing inconsistencies and enables direct prediction on new support ticket data.

Persisting the model in a reusable format supports future integration into operational systems, automated workflows, dashboards, and real-time SLA monitoring applications.


## Business interpretation
The final solution uses a Random Forest classifier with hyperparameters optimised through Grid Search and evaluated using five-fold cross-validation. A decision threshold of 0.50 was selected because it produced the strongest balance between precision and recall, achieving an F1-score of approximately 0.90. At this threshold, the model correctly identifies the majority of tickets that are likely to breach their SLA while maintaining a manageable number of false alarms. Operationally, this enables support managers to proactively prioritise high-risk tickets, allocate resources more effectively, and reduce SLA violations. Although some non-breaching tickets may be flagged unnecessarily, the business cost of these false positives is generally lower than the cost of missing genuine SLA breaches that could negatively affect customer satisfaction and service performance.
